# Single-Agent vs Multi-Agent

**Prerequisites:** from `02_agent_runtime`: `05_sub_agents.ipynb`

Chapter 2 already runs several agents at once. A `Think` calls `delegate`, the work splits into branches, and each branch runs its own loop with its own clean context. So before building anything, this notebook has to answer a plain question: **is that already a multi-agent system?**

The answer it arrives at is no. A branch is the same agent again with a narrower goal: the same system prompt, the same persona, the same tools. A multi-agent system is several agents that are *different*, because somebody designed each of them for its own job.

**By the end you'll have:**
- Seen, without calling a model, that a chapter 2 branch starts with its parent's system prompt, and that `delegate` has no field that could give it a different one
- Written an agent as a *definition*: a name, a persona, and the tools it may use
- Given one job that needs five different people to three setups: one agent, a chapter 2 fork, and five specialists
- Read what each produced, piece by piece, and measured what each cost
- A rule for when to use one agent, when a fork is ideal, and when the job needs specialists

## Setup

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "03_tools"):
    sys.path.append(str(HERE.parent / chapter))

import concurrent.futures
import json
import time
from dataclasses import dataclass, field

import llm
import think
from chat import Conversation, count_tokens
from runtime import Runtime
from observe import Observe
from act import Act
from registry import Registry

This notebook runs on **`openai/gpt-oss-20b`**, the smaller of the two gpt-oss models Groq serves. It is cheaper, it has its own daily quota, and a smaller model shows the strain of doing five jobs at once sooner than a bigger one does.

One adjustment is needed first. gpt-oss-20b stops at 2,048 output tokens unless told otherwise, and that count includes its hidden reasoning. On a long task it can spend the whole allowance thinking and return an empty answer. `Think` has no setting for this, so a small subclass adds one:

In [2]:
MODEL = "openai/gpt-oss-20b"


class Think(think.Think):
    def _sampling_kwargs(self):
        return {**super()._sampling_kwargs(), "max_tokens": 16000}

## A branch is a copy

When `Act` starts a branch, it builds the branch's conversation in `_seed_branch`. Chapter 2 chose what goes in on purpose: not the outer history, which made a branch answer other branches' questions, but *system messages plus the goal*.

That was the right fix for the problem chapter 2 had. It also means the thing that makes an agent who it is, its system prompt, is copied from the parent. Nothing here calls a model:

In [3]:
parent = Conversation(system="You write the recall response: a warm email to customers AND a formal notice to the regulator.")
parent.messages.append({"role": "user", "content": "We are recalling a batch of headphones. Write the response."})

branch = Act()._seed_branch(parent, {"id": "legal", "goal": "Write the notice to the regulator."})
for m in branch.messages:
    print(f"{m['role']:<7}{m['content']}")

fields = think._delegate_tool(parent)["function"]["parameters"]["properties"]["branches"]["items"]["properties"]
print("\nwhat delegate lets the model set per branch:", ", ".join(fields))

system You write the recall response: a warm email to customers AND a formal notice to the regulator.
user   Write the notice to the regulator.

what delegate lets the model set per branch: id, goal, allow_fork, explain, temperature, top_p, max_seconds, max_iterations, max_tool_calls, max_tokens


The branch meant to write a formal legal notice starts with a system prompt describing *both* jobs, the warm one included.

The list of fields is the other half. Per branch, the delegating model can set a goal, whether it may fork again, whether to explain itself, sampling (`temperature`, `top_p`) and budgets. There is no field for a system prompt and none for tools. `branch_runtime(branch)` does receive this dict, so a factory *could* build a different runtime per branch, but the dict holds nothing to build a different agent *from*. So chapter 2's factory builds the same agent every time.

That leaves one way for a branch to become someone in particular: the parent writes it into the `goal`, on the spot, in its own words.

## An agent is a definition

Chapter 1's `Agent` already had the right shape: `Agent(name, system, tools, ...)`. What it lacked was chapter 2's loop. Putting the two together takes a few lines:

In [4]:
@dataclass
class Agent:
    name: str
    persona: str                                    # its own system prompt
    tools: Registry = field(default_factory=Registry)

    def runtime(self):
        return Runtime(
            loop=[Think(model=MODEL, tools=self.tools.tools, allow_fork=False, explain=False),
                  Act(functions=self.tools.functions, schemas=self.tools.schemas),
                  Observe()],
            repeat_from=0,
            max_iterations=6,
        )

    def run(self, task):
        conversation = Conversation(system=self.persona)
        conversation.messages.append({"role": "user", "content": task})
        self.runtime().run(conversation)
        return conversation.messages[-1]["content"]

This is the chapter's idea in one class. A `persona` and a set of `tools` are fixed when the agent is *defined*, by you. A branch inherits both.

Nobody in this notebook needs a tool, so every agent here gets an empty `Registry`. The field is there because tools are half of what makes an agent different, and later notebooks use it.

## The job

A batch of headphones has a battery fault, and the company is recalling it. Answering that takes five pieces of writing, and in a real company five different people would write them. Each wants something different from the same facts:

| Who | Writes | For | What they care about |
|---|---|---|---|
| **Ana**, support lead | the email to customers | a worried owner | that they are safe and know exactly what to do |
| **Marcus**, PR director | the press statement | journalists | trust, proportion, nothing that invites a follow-up story |
| **Deborah**, product lawyer | the notice to the regulator | a regulator, and maybe a court | every material fact, and no admission of fault |
| **Raj**, lead engineer | the internal engineering brief | other engineers | the real numbers, the real uncertainty, the supplier named |
| **Zoe**, social media manager | the social post | someone scrolling | one action and one link, in two lines |

These pull against each other. Raj wants every number and Marcus wants as few as possible. Ana says sorry like a person and Deborah never says anything that sounds like fault. One brief of facts goes to everyone, including a section marked internal only.

In [5]:
BRIEF = """INCIDENT BRIEF: Aria 2 wireless headphones, battery overheating

What happened
- 14 customers have reported the headphones getting hot while charging; in 6 cases the battery visibly swelled.
- 2 customers suffered minor burns (one to a finger, one to the ear). No fires, no hospital stays.
- All 14 units come from one production batch: serial numbers AR2-2408-00001 to AR2-2408-03200, shipped 5-19 August. 3,200 units in total, about 2,900 already with customers.

What we know
- 11 of the returned units have been opened. In 9, the battery cell has a separator defect that lets it overheat while charging. In the other 2 the cause is not yet determined.
- 400 units from other batches were stress-tested. None showed the fault.
- Headphones are safe to use on battery; the risk is while charging.

What we are doing
- Voluntary recall of the affected batch, starting Monday 22 September.
- Customers should stop charging immediately, and choose a free replacement or a full refund at aria.example.com/recall. Prepaid return packaging is sent to them.
- The product-safety regulator must be notified within 2 working days.

INTERNAL ONLY -- never to appear in anything outside the company
- The cells came from our supplier Voltcell (lot VC-7731). Voltcell has not yet accepted responsibility.
- The 2 undetermined units could mean the fault is wider than the batch. Engineering puts this at low but not zero.
- One of the burned customers has a lawyer and has threatened to sue.
- Estimated recall cost: $410,000.

Use the facts in this brief exactly as they are given -- numbers, names, dates and the web address included -- wherever your piece needs them. Do not add any fact that is not here: no other names of people or companies, no other dates, years or numbers, no quotes, no phone numbers or email addresses, no causes or risks beyond those stated. If a piece would normally include something the brief does not give, leave it out."""

PERSONAS = {
    "customer email": """You are ANA, the customer support lead. You write to customers who own the product, and you write for the person reading at the kitchen table who is a bit scared. Your only goal is that they are safe and know exactly what to do next. You are warm and plain-spoken; you say sorry like a person, not like a company. You never hide a risk from a customer, and you never bury the instruction they need under explanations. You put the one action that matters first, then how to get their refund or replacement, step by step. You avoid jargon, legal phrasing and PR phrasing entirely. You would rather be clear than polished.""",

    "press statement": """You are MARCUS, the PR director. You write for journalists and the public, and your job is to protect the trust people have in the brand while being honest. You take responsibility in tone -- no defensive language, no blaming anyone -- and you keep things in proportion: this is one batch, the company found it and is acting fast. You write in confident, measured sentences that a journalist can quote. You never speculate, never give details that invite follow-up stories, and you always end by pointing people to where they can get help.""",

    "regulator notice": """You are DEBORAH, the company's product lawyer. You write to the product-safety regulator, and everything you write could be read out in court. You are formal, exact and complete: the regulator must get every material fact about the hazard, the affected units and the corrective action, because an incomplete notification is itself a violation. At the same time you never admit fault or legal liability, never use words like "defect caused injury" or "our negligence", and never speculate about causes that are not established. You state facts, numbers, dates and actions, in numbered paragraphs. Emotion and marketing have no place in what you write.""",

    "engineering brief": """You are RAJ, the lead hardware engineer. You write for other engineers inside the company, and you are blunt. Your readers need the real numbers, the real uncertainty and the real next steps, not reassurance. You say what is confirmed, what is not, and what could make this worse than it looks. You name suppliers, lots and failure modes precisely. You write in terse bullet points with headings. You never soften a finding to make it sound better, and you never pad.""",

    "social post": """You are ZOE, the social media manager. You write for people scrolling a feed, who will read two lines at most. You sound like a person, not a press office: human, direct, a little warm, never cute about a safety issue. Your posts are short -- well under 280 characters -- and always tell people the one thing to do and where to go. You never include detail that belongs in an email or a press release, and you never use hashtags on a safety post.""",
}

PARTS = {
    "customer email": "Write the email to customers who bought an affected Aria 2.",
    "press statement": "Write the press statement announcing the recall.",
    "regulator notice": "Write the notification to the product-safety regulator.",
    "engineering brief": "Write the internal engineering brief on the fault.",
    "social post": "Write the social media post announcing the recall.",
}

# The same five jobs, written as instructions for one writer rather than as five people. The one
# agent and the fork get these; the specialists get PERSONAS.
GUIDELINES = {
    "customer email": """Write for the customer who owns the product, reading at the kitchen table and a bit scared. The only goal is that they are safe and know exactly what to do next. Be warm and plain-spoken, and say sorry like a person, not like a company. Never hide a risk from a customer, and never bury the instruction they need under explanations. Put the one action that matters first, then how to get their refund or replacement, step by step. Avoid jargon, legal phrasing and PR phrasing entirely. Clear beats polished.""",

    "press statement": """Write for journalists and the public. The job is to protect the trust people have in the brand while being honest. Take responsibility in tone -- no defensive language, no blaming anyone -- and keep things in proportion: this is one batch, the company found it and is acting fast. Use confident, measured sentences a journalist can quote. Never speculate, never give details that invite follow-up stories, and always end by pointing people to where they can get help.""",

    "regulator notice": """Write to the product-safety regulator, knowing everything in it could be read out in court. Be formal, exact and complete: the regulator must get every material fact about the hazard, the affected units and the corrective action, because an incomplete notification is itself a violation. Never admit fault or legal liability, never use words like "defect caused injury" or "our negligence", and never speculate about causes that are not established. State facts, numbers, dates and actions, in numbered paragraphs. No emotion, no marketing.""",

    "engineering brief": """Write for other engineers inside the company, and be blunt. They need the real numbers, the real uncertainty and the real next steps, not reassurance. Say what is confirmed, what is not, and what could make this worse than it looks. Name suppliers, lots and failure modes precisely. Use terse bullet points with headings. Never soften a finding to make it sound better, and never pad.""",

    "social post": """Write for people scrolling a feed, who will read two lines at most. Sound like a person, not a press office: human, direct, a little warm, never cute about a safety issue. Keep it well under 280 characters, and always tell people the one thing to do and where to go. Never include detail that belongs in an email or a press release, and never use hashtags on a safety post.""",
}

TASK = ("We are recalling a batch of Aria 2 headphones. Write the full response: the customer email, the "
        "press statement, the regulator notice, the internal engineering brief and the social post.")

print(f"brief: {count_tokens(BRIEF)} tokens")
for piece, persona in PERSONAS.items():
    print(f"{piece:<18} persona: {count_tokens(persona)} tokens")

brief: 445 tokens
customer email     persona: 136 tokens
press statement    persona: 109 tokens
regulator notice   persona: 131 tokens
engineering brief  persona: 99 tokens
social post        persona: 101 tokens


## Three ways to do it

**One agent.** One writer, with one system prompt: *"You handle the company's written communications about a product recall"*, followed by instructions for each kind of piece and the brief. The instructions are the same five jobs the specialists do, written as rules for one writer rather than as five people, so nobody is named. It is asked for all five pieces in a single reply.

What goes where matters. The system prompt says *who the agent is*. The user message says *what to do now*, including how to lay out the answer. Only the system prompt ever reaches a branch.

**A chapter 2 fork.** The same system prompt as the one agent, one persona with the instructions for all five pieces and the brief, with a short request: *"Coordinate the company's written response to a product recall. Write the full response: the customer email, the press statement, the regulator notice, the internal engineering brief and the social post. Split the work between five sub-agents, one per piece."* Each branch is built the way chapter 2 builds branches: the parent's system prompt, plus the goal the parent writes.

**Five specialists.** Five `Agent`s, each with one persona plus the brief, each asked for its own piece. They run side by side and their pieces are simply collected. How a supervisor would decide who does what is notebook 2. Here the split is wired by hand, so that the one thing that differs between the three setups is *who* does the writing.

In [6]:
ALL_FIVE = ("You handle the company's written communications about a product recall. Each kind of piece "
            "has its own instructions.\n\n" +
            "\n\n".join(f"{piece.upper()}\n{text}" for piece, text in GUIDELINES.items()) +
            f"\n\n{BRIEF}")

# 1. one agent. How to lay out the answer is about this request, not about who the agent is, so it
#    goes in the user message. A branch never sees its parent's user message; it would see this if
#    it sat in the system prompt.
one_agent = Agent("recall_team", ALL_FIVE)
ONE_AGENT_TASK = TASK + " Output the five pieces, each headed on its own line by its name in capitals."

# 2. a chapter 2 fork. Every branch comes from this one factory, whatever it was asked to write.
#    `written` keeps what each branch produced, so the pieces can be read one by one afterwards.
written = {}

class Clone:
    def __init__(self, branch):
        self.branch = branch
        self.inner = Agent("clone", "").runtime()

    def run(self, conversation):
        # Act gives every branch runtime a way to be cancelled; pass it on to the real one
        self.inner.cancel = getattr(self, "cancel", None)
        self.inner.run(conversation)
        written[self.branch.get("id") or self.branch["goal"][:30]] = (
            self.branch["goal"], conversation.messages[-1]["content"])
        return conversation

def forking_runtime():
    return Runtime(
        loop=[Think(model=MODEL, explain=False), Act(branch_runtime=Clone), Observe()],
        repeat_from=0,
        max_iterations=3,
    )

# the fork has the same system prompt as the one agent; its request is one line, and how to split
# the work, if at all, is left to it
FORK_TASK = ("Coordinate the company's written response to a product recall. Write the full response: "
             "the customer email, the press statement, the regulator notice, the internal engineering "
             "brief and the social post. Split the work between five sub-agents, one per piece.")

# 3. five specialists
specialists = [Agent(piece, f"{persona}\n\n{BRIEF}") for piece, persona in PERSONAS.items()]

The fork stops after three steps: the parent decides to delegate, the branches run, and their results come back. Left to go on, the parent's next step would be to read all five pieces back and type them out again as its own answer. In an earlier run, that one request was bigger than the 8,000 tokens per minute this account is allowed on gpt-oss-20b, and Groq refused it outright. The pieces are already written by then, so nothing is lost by stopping.

## What each agent starts with

Every agent's first request is its system prompt plus its task, and every later call carries at least that much again. `Think._build_messages()` returns what would be sent without sending it, so it can be counted for free. A branch's task is the goal the parent writes, which does not exist until the fork runs, so its row shows only what it gets before that.

In [7]:
def first_request(system, task, allow_fork=False):
    conversation = Conversation(system=system)
    conversation.messages.append({"role": "user", "content": task})
    messages = Think(model=MODEL, allow_fork=allow_fork, explain=False)._build_messages(conversation)
    tools = [think._delegate_tool(conversation)] if allow_fork else []
    return (sum(count_tokens(m["content"]) for m in messages if m.get("content"))
            + (count_tokens(json.dumps(tools)) if tools else 0))

rows = [("one agent", first_request(ALL_FIVE, ONE_AGENT_TASK), ""),
        ("fork: the parent", first_request(ALL_FIVE, FORK_TASK, allow_fork=True), ""),
        ("fork: each branch", first_request(ALL_FIVE, ""), "  + the goal the parent writes")]
rows += [(f"specialist: {a.name}", first_request(a.persona, PARTS[a.name]), "") for a in specialists]

for name, tokens, note in rows:
    print(f"{name:<32}{tokens:>6} tokens{note}")

one agent                         1076 tokens
fork: the parent                  1640 tokens
fork: each branch                 1021 tokens  + the goal the parent writes
specialist: customer email         658 tokens
specialist: press statement        625 tokens
specialist: regulator notice       649 tokens
specialist: engineering brief      616 tokens
specialist: social post            618 tokens


- **The one agent starts with 1,076 tokens:** one persona, the instructions for all five pieces, the brief, and the request.
- **The fork's parent starts with 1,640:** the same, plus the `delegate` schema and the guidance on when to use it.
- **A fork branch starts with 1,021 before its goal is added:** the parent's whole system prompt, all five sets of instructions included, to write *one* piece.
- **A specialist starts with about 620 to 660:** its own instructions, the brief, and its piece. Everything it carries is about the one piece it writes.

## Running all three

A `Meter` wraps the one client every component shares, so each run's calls and tokens come from the API's own `usage` field rather than an estimate. It also notes which task each call was working on, which is how the fork's branches get told apart afterwards.

In [8]:
class Meter:
    def __enter__(self):
        self.calls = []
        real = self._real = llm.client.chat.completions.create

        def create(*args, **kwargs):
            response = real(*args, **kwargs)
            self.calls.append({
                "asked": next((m["content"] for m in kwargs.get("messages", [])
                               if m.get("role") == "user"), ""),
                "prompt": response.usage.prompt_tokens,
                "completion": response.usage.completion_tokens,
            })
            return response

        llm.client.chat.completions.create = create
        self._started = time.monotonic()
        return self

    def __exit__(self, *exc):
        self.seconds = time.monotonic() - self._started
        llm.client.chat.completions.create = self._real

    def summary(self):
        prompt = sum(c["prompt"] for c in self.calls)
        completion = sum(c["completion"] for c in self.calls)
        return {"calls": len(self.calls), "prompt": prompt, "completion": completion,
                "total": prompt + completion, "seconds": round(self.seconds)}

costs = {}

### One agent

In [9]:
with Meter() as meter:
    together = one_agent.run(ONE_AGENT_TASK)
costs["one agent"] = meter.summary()
print(together)
print("\n", costs["one agent"])

CUSTOMER EMAIL  
Subject: Important safety information – take action now  

Hi,

If you own Aria 2 headphones, please **stop charging them immediately**.  

We have found that the battery can overheat and swell in some units, and a few customers have suffered minor burns.  
We’re offering a free replacement or a full refund.  

**What to do right away:**  

1. Unplug the headphones from the charger and keep them in a safe, well‑ventilated spot.  
2. Go to aria.example.com/recall.  
3. Choose “Replacement” or “Refund” and fill out the form.  
4. We’ll send you a prepaid return package.  

After you return the unit, we’ll ship you a new pair or process your refund.  

I’m really sorry you’re dealing with this. We’re on it and want to make things right.  

Thank you for your patience and safety.  

—  
Aria Support Team

---

PRESS STATEMENT  
Date: 22 September 2024  

Aria is recalling a specific batch of Aria 2 headphones (serial numbers AR2‑2408‑00001 to AR2‑2408‑03200, shipped 5‑19 A

### A chapter 2 fork

In [10]:
forked = Conversation(system=ALL_FIVE)
forked.messages.append({"role": "user", "content": FORK_TASK})
with Meter() as fork_meter:
    forking_runtime().run(forked)
costs["fork"] = fork_meter.summary()

for branch_id, (goal, text) in written.items():
    print(f"=== [{branch_id}]\ngoal: {goal}\n\n{text}\n")
print(costs["fork"])

=== [press_statement]
goal: Write a press statement for journalists and the public. Take responsibility, no defensive language, no blaming. State that the issue is in one batch, company found it, is acting fast. Use confident, measured sentences. No speculation, no details that invite follow‑up. End by pointing people to aria.example.com/recall for help. Use facts: 14 customers, 6 swollen batteries, 2 minor burns, batch AR2‑2408‑00001‑AR2‑2408‑03200, 3,200 units, shipped 5‑19 Aug, recall starts Monday 22 September, stop charging immediately, etc.

**Aria Announces Recall of Aria 2 Wireless Headphones**

Aria today announced a voluntary recall of the Aria 2 wireless headphones that were shipped in batch AR2‑2408‑00001 through AR2‑2408‑03200, a total of 3,200 units distributed between 5 and 19 August.

A total of 14 customers have reported the headphones becoming hot while charging, and 6 of those units showed visibly swollen batteries. Two customers sustained minor burns—one to a finger

### Five specialists

In [11]:
with Meter() as meter:
    with concurrent.futures.ThreadPoolExecutor() as pool:
        pieces = dict(zip(PERSONAS, pool.map(lambda a: a.run(PARTS[a.name]), specialists)))
costs["specialists"] = meter.summary()

for piece, text in pieces.items():
    print(f"=== {piece}\n\n{text}\n")
print(costs["specialists"])

=== customer email

Subject: Important safety notice for your Aria 2 headphones

Hi there,

We’re writing because we learned that some Aria 2 headphones can get hot while charging and, in some cases, the battery can swell or cause a burn. This only affects the batch that shipped between 5–19 August, serial numbers AR2‑2408‑00001 to AR2‑2408‑03200 (about 2 900 units are already with customers).

**What you need to do right now**

1. **Stop charging the headphones immediately.**  
   If they’re already plugged in, unplug them and let them cool.

2. **Choose a free replacement or a full refund** by visiting  
   <https://aria.example.com/recall>

3. We will send you a prepaid return envelope.  
   Just put the headphones in the box and drop it off at any post office or courier.

4. Once we receive your unit, you’ll get either a replacement Aria 2 or a full refund.

**Why this matters**

All 14 units that have been returned show a battery separator defect that can cause overheating while c

## Reading what came back

These are single runs, so read them as examples, not scores. Every setup makes ordinary slips: invented dates (2024 appears in the one agent's pieces and the fork's), an invented hotline in the fork's email, *"Forty‑zero units"* and an invented company name from the specialists. The brief's instruction not to invent anything cut these down but did not stop them.

**The one agent is the clear loser.** It wrote all five pieces in one reply of 1,003 output tokens, and the pieces bled into each other:

- The **press statement** tells journalists the company is *"working with our supplier, Voltcell, to resolve the issue."* The supplier is under INTERNAL ONLY in the brief, and its responsibility is in dispute.
- The **regulator notice** lists *"7. Supplier of affected cells: Voltcell, lot VC‑7731"* and *"9. Estimated recall cost: $410,000."* Both are internal.
- The **social post** ends *"No fire risk, just battery heat."* Nothing in the brief says that. It plays down a safety risk in the one piece most people will read.
- The **engineering brief** says *"Two customers with legal representation"*. It was one.

**The fork did about as well as the specialists.** Its regulator notice and press statement keep every internal fact out. Its engineering brief is blunt, with the supplier, the dispute and the cost where they belong. Its email leads with the action. One real weakness: the regulator notice leaves out the investigation (9 of 11 units with a separator defect, 2 undetermined, 400 tested clean). The instructions call that material, and a regulator would expect it.

**The specialists** wrote in voice, with the brief's facts, and kept internal facts out of the public pieces, as in their saved run above.

This was not the only run. Across every run made while building this notebook, the regulator notice carried an internal fact in **4 of 6** runs of the one agent, **2 of 6** of the fork (set up in slightly different ways over those runs), and **1 of 4** of the specialists. The numbers are small, but the one agent is consistently the worst, and the fork and the specialists are close.

**Why the fork recovered most of the quality:** each branch had one piece to write, in a fresh context. That is what the one agent lacked. It had to hold five audiences in one reply, and each piece got a fifth of its attention.

But the fresh context is only half of it. The other half is that the parent, on its own initiative, copied each piece's rules and facts into that branch's goal. The next section shows why that matters so much.

## What the branches were given

Every branch was started by `_seed_branch`: the parent's system prompt, then its goal. Here is what that system prompt held, and for each goal, whether it named the person writing and which of the brief's key facts it repeated:

In [12]:
fork = next(m for m in forked.messages if m.get("type") == "fork")
system = Act()._seed_branch(forked, fork["branches"][0]).messages[0]["content"]
print(f"every branch's system prompt holds all five sets of instructions: "
      f"{all(g in system for g in GUIDELINES.values())}, "
      f"and the brief: {BRIEF in system}\n")

NAMES = ("ANA", "MARCUS", "DEBORAH", "RAJ", "ZOE")
FACTS = {"serial range": "AR2-2408", "recall link": "aria.example.com/recall", "start date": "22 Sep",
         "the burns": "burn", "9 of 11": "9 of", "supplier": "Voltcell"}
for b in fork["branches"]:
    goal = b["goal"].replace("\u2011", "-")
    named = [n for n in NAMES if n.lower() in goal.lower()]
    carried = [name for name, mark in FACTS.items() if mark.lower() in goal.lower()]
    print(f"[{b.get('id', '?')}] goal of {count_tokens(b['goal'])} tokens; names {', '.join(named) or 'no one'}; "
          f"facts repeated: {', '.join(carried) or 'none'}")

every branch's system prompt holds all five sets of instructions: True, and the brief: True

[customer_email] goal of 134 tokens; names no one; facts repeated: serial range, recall link, start date, the burns
[press_statement] goal of 129 tokens; names no one; facts repeated: serial range, recall link, start date, the burns
[regulator_notice] goal of 131 tokens; names no one; facts repeated: serial range, recall link, start date, the burns
[engineering_brief] goal of 158 tokens; names no one; facts repeated: serial range, start date, the burns, supplier
[social_post] goal of 65 tokens; names no one; facts repeated: recall link, start date


Every branch's system prompt holds the parent's whole prompt: all five sets of instructions and the brief. No one is named in it, so there is no persona for a goal to point at. The parent wrote goals of 65 to 158 tokens, and each one restates the instructions for its piece and the facts it needs:

> *"Write a formal, exact regulator notice. Include every material fact: hazard, affected units, corrective action. Do not admit fault or liability…"*

So a branch is the parent copied, handed a smaller problem with a well-written brief for it.

**That is what kept the voices apart, and it was the parent's choice, not the fork's design.** Look at what a branch would have without it. Its system prompt holds the rules for all five pieces side by side: be warm, be formal, be blunt, never name the supplier, always name the supplier. A goal that only says *"write the regulator notice"* tells it which piece, but leaves it to sort out which of those rules are its own, with the other four sets right there in its prompt. The five voices share one prompt, and they mix.

An earlier version of this notebook shows exactly that. Its goals named the piece and little else, *"Write a formal regulator notice in the voice of DEBORAH…"*. The branch that wrote the regulator notice listed the supplier, the supplier's dispute, the cost and the engineer's private worry about a wider fault. That is the engineer's content, taken from the prompt the two branches shared.

So a fork's quality rests on the parent getting each goal right: the rules for that piece, and the facts it needs, written out at run time in the parent's own words. When it does, a branch behaves like a specialist. When it doesn't, the branch falls back on a system prompt that was written for all five pieces at once. Nothing in the fork checks which of those happened.

It can go further wrong still. A version of this notebook gave the parent a one-line system prompt and put the brief only in its request, so branches inherited no facts at all. That parent wrote one-line goals with none of the brief's facts in them, and its branches invented a different incident: a regulator notice about headphones *"igniting"*, a press statement saying *"No injuries have been reported"*, an engineering brief counting *"1,047,832 units"*.

A specialist does not depend on any of this. Its rules and facts were written once, by you, before the run, and it has no other piece's rules to confuse them with.


## What it cost

In [13]:
print(f"{'':<14}{'calls':>6}{'prompt':>9}{'completion':>12}{'total':>8}{'seconds':>9}")
for name, c in costs.items():
    print(f"{name:<14}{c['calls']:>6}{c['prompt']:>9}{c['completion']:>12}{c['total']:>8}{c['seconds']:>9}")

               calls   prompt  completion   total  seconds
one agent          1     1153        1003    2156        3
fork               6     7727        6367   14094       49
specialists        5     3561        3628    7189       23


- **The one agent was cheapest by far:** one call, 2,156 tokens, 3 seconds.
- **The specialists cost 3.3× as much:** five copies of the brief, and a full piece from each.
- **The fork cost 6.5× as much, about twice the specialists.** Each branch starts with about 400 more tokens than a specialist, since it carries the instructions for all five pieces, and five branches make that about 2,000. The parent's call carries the `delegate` schema. Its goals are long, because it restated each piece's instructions and facts. And its branches wrote longer pieces: 6,367 output tokens against the specialists' 3,628. It was also stopped before its final answer, which would have been the biggest request of the run.

The seconds are noisy: Groq is fast, and both the fork and the specialists back off when five agents at once hit the 8,000 tokens-per-minute cap.

So the fork bought most of the specialists' quality, at about twice their price. It got that quality because the parent happened to copy each piece's rules into its goal. A parent that didn't would have paid the same and got mixed voices.

## Which one to use

| | Use it when | What it gives you | What it costs |
|---|---|---|---|
| **One agent** | the job is small, and every part of it wants the same person | the cheapest answer | every part shares one reply and one context, so parts get thin and bleed into each other |
| **Fork** | the job is **too big for one context**, but **every part wants the same person** | each part gets a fresh, short context and one problem to solve | the parent's system prompt in every branch; if the parts need different rules, the result depends on the parent writing each part's rules into its goal |
| **Specialists** | the parts of the job need **different people** | each agent carries only its own instructions, fixed in advance by you | you design and maintain each one |

**A fork is ideal when the job is bigger than one context should hold, but one persona can do every part of it.** Reviewing forty files against the same coding standard. Summarising ten long reports for the same reader. Checking each clause of a contract with the same lens. Researching five competitors the same way. Each branch is the same agent, copied, with one piece of the work and a context that holds only that piece. Nothing in the parent's system prompt is wrong for any branch, because every branch is meant to be the same person. That is where copying costs nothing you would want to change, and the fresh, short context is pure gain. In this notebook, that is what lifted the fork above the one agent.

**Specialists are right when the parts need different people:**

- **Different voices.** Warm, formal, blunt and casual in one job.
- **Goals that pull against each other.** The engineer wants every number; the press statement wants as few as possible.
- **Things one of them must say and another must not.** The supplier's name belongs in the engineering brief and nowhere else.
- **Different tools.** Nobody here needed one, but an agent that must not be able to send an email or move money is a reason to split on its own.

A fork can imitate specialists, as it did here, but only when the parent writes each branch a goal that restates that part's rules and facts. That is the parent improvising a specialist at run time, in its own words, every time, and nothing checks that it did. When it doesn't, every branch writes from one system prompt meant for all the parts, and the voices mix. Specialists fix each part's rules once, in advance, carry nothing else, and cost less per piece.

Some reasons that are not enough on their own:

- **Context isolation.** Forks and specialists both have it. It is a reason to split the work, not a reason to use different agents.
- **Parts that can run in parallel.** Branches already do that.

The specialists here were wired up by hand: five agents, each given its piece. Notebook 2 gives them a supervisor that reads the job and decides which agent gets what. A chapter 2 parent cannot do that, because it has no list of agents to choose from.